# 01 — Exploratory Data Analysis (Phase 1)

**Project:** AI-Powered Cybersecurity Awareness Assistant — binary URL phishing classification.

This notebook is the written walkthrough of the EDA. The analysis itself is produced by the
reproducible DVC stage `eda` (`src/eda/run_eda.py`), which writes the figures to `reports/phase1/eda/`
and the numbers to `reports/phase1/eda_summary.json`. The notebook only reads those outputs, so the
numbers here always match the current `dvc.lock`.

To regenerate: `dvc repro eda`, then
`jupyter nbconvert --to notebook --execute notebooks/01_eda.ipynb --output 01_eda.ipynb`
(add `--inplace` to keep outputs locally; the committed copy has no outputs).

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = ROOT / "reports" / "phase1"
EDA = REPORTS / "eda"
summary = json.loads((REPORTS / "eda_summary.json").read_text())
prepare = json.loads((REPORTS / "prepare_summary.json").read_text())
validation = json.loads((REPORTS / "validation.json").read_text())
pct = lambda x: f"{100 * x:.1f}%"


## 1. Datasets, shape, data types and missing values

| Dataset | Role |
|---|---|
| PhiUSIIL | training pool + normal held-out test (undated) |
| PhishVN v3.1.0 (gold+silver) | evaluation only: cross-source + time-stratified (within PhishVN) |
| SMSSpamCollection | versioned for provenance only, not used (spam ≠ phishing) |

In [ ]:
for name in ("phiusiil_raw", "phishvn_raw"):
    p = summary[name]
    display(Markdown(f"**{name}** — shape {p['shape']} (of {p['n_columns_total']} columns in the file), "
                     f"exact duplicate rows: {p['exact_duplicate_rows']}"))
    display(pd.DataFrame({"dtype": p["dtypes"], "missing": pd.Series(p["missing_values"])}).fillna(0))


PhishVN's missing values are expected: `collected_at` is empty for sources that attest no event date
(registry/top-list benign), and several columns (`text`, `msg_len`, …) belong to channels not included in
this open release.

## 2. Class balance

In [ ]:
display(Image(EDA / "01_class_balance.png"))
display(pd.Series(summary["phiusiil_class_counts"], name="PhiUSIIL rows"))
display(pd.DataFrame(summary["phishvn_source_tier_label_counts"]).pivot_table(index=["source", "tier"], columns="label", values="n", fill_value=0))


PhiUSIIL is moderately imbalanced (≈57% legitimate). In PhishVN, the verified gold/silver core is
dominated by legitimate rows, while most phishing rows are in the bronze community tier, which we exclude
(community labels, ~12% audited label noise, reconstructed dates).

## 3. Collection-format artefacts — the most important finding

Character models learn whatever separates the classes. If the classes were collected in different
*formats*, the model learns the format.

In [ ]:
display(Image(EDA / "02_format_artefacts.png"))
display(Markdown("**PhiUSIIL, raw URL string**"))
display(pd.DataFrame(summary["format_artefacts_raw_phiusiil"]).T)
display(Markdown("**PhishVN, raw URL string**"))
display(pd.DataFrame(summary["format_artefacts_raw_phishvn"]).T)
display(Markdown("**After canonicalisation (scheme, leading www. and trailing slash removed), share of rows with a path:**"))
display(pd.Series(summary["has_path_rate_after_canonicalisation"]))


**Interpretation.**
- In raw PhiUSIIL, every legitimate URL starts with `https://www.` and has no path or trailing slash.
  Phishing URLs are a mix. A raw-URL model would score highly by learning `http://` and `/`, not phishing.
  The pipeline therefore canonicalises every URL in every dataset (see the examples below).
- A **residual path shortcut** remains: legitimate PhiUSIIL URLs are home pages only, so "has a path" is
  only true for phishing. This cannot be removed without discarding real information. It is reported
  explicitly (the reference rule in `reports/phase1/model_comparison.md`) and is one reason held-out
  scores must be read together with the shift and temporal results.
- PhishVN URLs are mostly bare hostnames (no paths), which is a genuine distribution shift.

In [ ]:
for label, rows in summary["canonicalisation_examples"].items():
    display(Markdown(f"**{label}**"))
    display(pd.DataFrame(rows))


## 4. URL length

In [ ]:
display(Image(EDA / "03_url_length.png"))
for name, stats in summary["model_text_length"].items():
    display(Markdown(f"**{name}** (characters of canonical URL)"))
    display(pd.DataFrame(stats).T)


## 5. Top-level domains

In [ ]:
display(Image(EDA / "04_top_tlds.png"))
display(Markdown(f"P(legitimate | `.vn` TLD) in PhishVN shift_test: **{summary['phishvn_shift_p_legitimate_given_vn_tld']}**"))


TLD composition differs strongly between datasets. In PhishVN, `.vn` is mostly legitimate, which is
a known shortcut documented by the dataset authors. A PhiUSIIL-trained model has seen few `.vn` domains,
so this is part of what the shift evaluation measures.

## 6. Duplicates, label conflicts and domain groups

In [ ]:
display(pd.DataFrame(summary["cleaning"]).T[[c for c in ["rows_raw", "dropped_too_short", "label_conflict_texts",
    "label_conflict_rows_dropped", "duplicate_rows_dropped", "removed_exact_text_overlap", "removed_domain_overlap", "rows_final"]
    if c in pd.DataFrame(summary["cleaning"]).T.columns]])
display(Image(EDA / "05_domain_group_sizes.png"))
display(pd.Series({k: v for k, v in summary["domain_groups"].items() if k != "largest_groups"}))
display(pd.Series(summary["domain_groups"]["largest_groups"], name="rows in largest groups"))


Duplicates are measured on the **model input** (canonical URL). Texts carrying both labels are
dropped entirely. The split assigns whole **registrable domains** to train/val/test, so the same domain
or campaign never appears on both sides. Hosting platforms (`*.web.app`, `*.firebaseapp.com`, …) are
split per tenant using the Public Suffix List's private section; otherwise one platform would form a
single group of thousands of phishing rows and distort the class balance between splits.

In [ ]:
display(pd.Series(summary["prepared_rows"], name="rows"))
display(pd.Series(summary["prepared_phishing_rate"], name="phishing rate"))
display(Markdown(f"Validation stage: **{validation['status']}** ({validation['checks_total']} checks, {validation['checks_failed']} failed)"))


## 7. Time: source-attested dates in PhishVN

In [ ]:
display(Image(EDA / "06_phishvn_timeline.png"))
display(pd.DataFrame(summary["phishvn_attested_dates"]).T)
display(Markdown("Dates present in the file but **not used** (not source-attested / excluded tier):"))
display(pd.DataFrame(summary["phishvn_dates_not_used"]))


Only NCSC detection dates (phishing) and NCSC certification dates (trusted organisations) are used.
`scraped_at` is a fetch time and the ChongLuaDao dates were reconstructed after the fact, so neither is
used. PhiUSIIL has no row-level dates (the 2024-03-08 timestamp of its file is not a collection date), so
no claim is made that PhishVN is newer than the training data. The dashed line marks the split date
used to stratify PhishVN by time; see `docs/EVALUATION_METHODOLOGY.md`.

## 8. What does the PhishVN positive label contain?

In [ ]:
display(Image(EDA / "07_phishvn_abuse_types.png"))
display(pd.Series(summary.get("phishvn_verified_phishing_abuse_types", {}), name="verified phishing rows"))
display(Image(EDA / "08_phishvn_scenarios.png"))


According to the PhishVN datasheet, `phishing` means *listed by a Vietnamese anti-fraud feed*.
Credential phishing is one category among gambling, investment and counterfeit-shop fraud. Most rows
cannot be typed from the URL (`unknown`). Recall on PhishVN should therefore be read as fraud-URL recall.

## 9. Summary of EDA findings → pipeline decisions

| Finding | Decision |
|---|---|
| Scheme / `www.` / trailing-slash format separates the PhiUSIIL classes | Canonicalise all URLs identically |
| Legitimate PhiUSIIL URLs never have paths | Report the "has path" reference rule; read held-out scores with care |
| Hosting platforms create huge domain groups | Group by registrable domain using PSL private suffixes |
| Duplicates and label conflicts after canonicalisation | Deduplicate on model input; drop conflicting texts |
| PhiUSIIL has no dates; PhishVN has attested dates for two sources | PhishVN is evaluation-only; recall stratified by PhishVN detection date (no train-past/test-future claim) |
| PhishVN bronze tier is noisy and its dates are reconstructed | Use gold+silver only |
| SMS data labels spam, not phishing | Excluded (`task.include_sms: false`) |
